# Notebook 04: Fine-Tuning with LoRA

This is the second way to adapt a model, and the one most people mean when they say "training". Here we actually change the model itself so it picks up our domain, rather than just handing it documents to read.

## What fine-tuning is

Fine-tuning means taking a model that has already been trained, and training it a bit more on your own data. We are not starting from nothing. Training a model from scratch takes months and a fortune in computing. Instead we begin with a capable model and nudge it toward our area.

A good way to picture it: you hire an experienced analyst and spend a week showing them how your firm does things. You are not re-teaching them to read. You are giving them your context, your style, and your specifics.

## The catch: doing it the full way is expensive

Training every single weight in a model is heavy. A 7 billion parameter model needs roughly 28 GB of memory just to hold, before you add the extra memory that training itself requires. That is out of reach for most laptops and plenty of servers too.

This is the problem LoRA solves.

## LoRA, the shortcut

LoRA stands for Low-Rank Adaptation. It rests on a neat finding: the changes you need to specialise a model are small and sit in a tiny corner of the maths. You do not have to touch most of the weights.

So instead of editing a giant weight grid directly, LoRA freezes it and bolts on two much smaller grids beside it. Only those small grids get trained.

```
Normal weight grid W:   4096 x 4096  =  about 16 million numbers   (frozen, left alone)

LoRA instead trains two small grids:
   A is 4096 x 16
   B is 16 x 4096
   together: about 131 thousand numbers   (roughly 120 times fewer to train)
```

The "16" there is the rank, the one knob that sets how big the add-on grids are. Because the original model stays frozen and only the small add-ons change, the memory and time needed drop enormously. That is what lets us train on a laptop.

## Adapt this

To train on your own data, just load the `instruction_dataset` you built in notebook 02 in place of the demo data. The LoRA settings and the training loop are exactly the same for any instruction dataset.


In [ ]:
import torch
from pathlib import Path
from datasets import load_from_disk, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer

PROCESSED_DIR = Path("../data/processed")
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)

device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Using device: {device}")

## 1. Load the base model and tokeniser

First we load the same TinyLlama model and its tokeniser. One small detail in the next cell: we load it in the full-precision number format rather than the half-precision one. On Apple Silicon, training in half precision can get unstable, and full precision is safer here.


In [ ]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokeniser = AutoTokenizer.from_pretrained(MODEL_ID)

# The tokeniser needs a padding token to handle variable-length sequences in a batch.
# TinyLlama doesn't have one by default, so we use the end-of-sequence token as padding.
if tokeniser.pad_token is None:
    tokeniser.pad_token = tokeniser.eos_token

# Load in float32 for training stability on MPS
# (float16 training can have instability on Apple Silicon)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
)
model = model.to(device)

# Count parameters before LoRA
total_params = sum(p.numel() for p in model.parameters())
print(f"Base model loaded: {total_params/1e9:.2f}B parameters")

## 2. Setting up LoRA

LoRA has a few settings worth understanding before we switch it on:

- **`r` (the rank).** How big the add-on grids are. Bigger means more learning capacity but more to train. Common values run from 4 to 64. We use 16.
- **`lora_alpha`.** A scaling factor that controls how strongly the add-ons affect the model. The usual rule of thumb is to set it to twice the rank.
- **`target_modules`.** Which parts of the model get the add-ons. The attention projection layers (the `q_proj` and `v_proj` we spotted back in notebook 01) are the standard choice.
- **`lora_dropout`.** A small bit of randomness during training that helps stop the model from simply memorising a tiny dataset.


In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,   # We're fine-tuning a causal language model
    r=16,                            # Rank - number of dimensions in adapter matrices
    lora_alpha=32,                   # Scaling factor (typically 2 x r)
    lora_dropout=0.1,                # Dropout on LoRA layers for regularisation
    target_modules=[                 # Which attention layers to apply LoRA to
        "q_proj",                    # Query projection
        "v_proj",                    # Value projection
        "k_proj",                    # Key projection
        "o_proj",                    # Output projection
    ],
    bias="none",                     # Don't add LoRA to bias parameters
)

# Wrap the model with LoRA - this inserts the adapter matrices
model = get_peft_model(model, lora_config)

# Show parameter breakdown
model.print_trainable_parameters()

In [ ]:
# Visual inspection: let's see what the LoRA-wrapped model looks like
# We can see the original frozen layers alongside the small trainable adapters

print("LoRA adapter structure (first attention layer):")
print()
for name, module in model.model.layers[0].self_attn.named_modules():
    if name:  # skip root
        n_params = sum(p.numel() for p in module.parameters())
        trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
        print(f"  {name:40s}  params: {n_params:,}  trainable: {trainable:,}")

## 3. Preparing the data

The model cannot train on text directly. Just like in notebook 01, the text has to be turned into tokens (numbers) first. The next cells load our dataset and run it through the tokeniser so it is ready for training.


In [ ]:
import sys
sys.path.append("..")
from utils.templates import format_instruction

# Load dataset from Notebook 02, or create a minimal one for demonstration
dataset_path = PROCESSED_DIR / "instruction_dataset"
if dataset_path.exists():
    split = load_from_disk(str(dataset_path))
    train_data = split["train"]
    val_data = split["test"]
    print(f"Loaded dataset: {len(train_data)} train, {len(val_data)} val")
else:
    # Minimal fallback dataset for demonstration
    examples = [
        {"instruction": "What was Apple's total net sales for fiscal year 2023?",
         "context": "Apple reported total net sales of $383.3 billion for fiscal 2023.",
         "response": "Apple's total net sales for fiscal year 2023 were $383.3 billion."},
        {"instruction": "How did Microsoft Azure grow in fiscal 2023?",
         "context": "Azure and cloud services revenue grew 27% in fiscal year 2023.",
         "response": "Microsoft Azure grew 27% in fiscal 2023, outpacing overall company growth."},
        {"instruction": "What drove JPMorgan Chase's strong 2023 net income?",
         "context": "JPMorgan reported net income of $49.6 billion, driven by higher net interest income as rates rose.",
         "response": "JPMorgan's strong 2023 net income of $49.6B was primarily driven by higher net interest income in a rising rate environment."},
        {"instruction": "What percentage of Apple's revenue comes from international markets?",
         "context": "International markets accounted for 58% of total Apple revenues in fiscal 2023.",
         "response": "International markets accounted for 58% of Apple's total revenues in fiscal 2023."},
    ]
    for ex in examples:
        ex["text"] = format_instruction(ex["instruction"], ex["response"], ex["context"])
    train_data = Dataset.from_list(examples)
    val_data = Dataset.from_list(examples[:2])
    print(f"Using demonstration dataset: {len(train_data)} train, {len(val_data)} val")

In [ ]:
MAX_LENGTH = 512  # Maximum token length - truncate anything longer

def tokenise(examples):
    """
    Tokenise a batch of examples.
    The 'text' column contains the full formatted instruction string.
    """
    return tokeniser(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",  # Pad shorter sequences to MAX_LENGTH
    )

# Apply tokenisation to both splits
tokenised_train = train_data.map(tokenise, batched=True, remove_columns=train_data.column_names)
tokenised_val   = val_data.map(tokenise, batched=True, remove_columns=val_data.column_names)

# Set the format to PyTorch tensors
tokenised_train.set_format("torch")
tokenised_val.set_format("torch")

print(f"Train: {len(tokenised_train)} examples, each with {MAX_LENGTH} tokens")
print(f"Columns: {tokenised_train.column_names}")

## 4. Training

Here is the actual training loop, the same one you saw built by hand in notebook 01b:

1. Show the model a batch of examples.
2. At each spot, the model guesses the next token.
3. Compare its guesses to the real next tokens and score how wrong it was (the loss).
4. Work out which way to adjust the weights to be less wrong (the gradients).
5. Nudge the LoRA add-ons in that direction. The frozen base model is left alone.
6. Repeat.

We do not write those steps out by hand here. Hugging Face's `Trainer` handles them. We just hand it the settings.


In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR / "tinyllama-financial-lora"),
    
    # Training duration
    num_train_epochs=3,              # How many full passes through the training data
    
    # Batch size - how many examples per gradient update
    # On a 16GB Mac, we use a small batch + gradient accumulation
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   # Effective batch size = 2 x 4 = 8
    per_device_eval_batch_size=2,
    
    # Optimisation
    learning_rate=2e-4,              # How fast to update weights. LoRA uses higher LR than full fine-tuning.
    warmup_ratio=0.1,                # Gradually increase LR for first 10% of training
    weight_decay=0.01,               # L2 regularisation to prevent overfitting
    
    # Logging and checkpointing
    logging_steps=10,
    eval_strategy="epoch",           # Evaluate on validation set after each epoch
    save_strategy="epoch",
    load_best_model_at_end=True,
    
    # Apple Silicon - use MPS
    use_mps_device=(device == "mps"),
    
    # Precision - fp32 for stability on MPS
    fp16=False,
    bf16=False,
)

# Data collator: handles padding and preparing batches
# mlm=False because we're doing causal language modelling (predict next token), not masked
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokeniser,
    mlm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenised_train,
    eval_dataset=tokenised_val,
    tokenizer=tokeniser,
    data_collator=data_collator,
)

print("Trainer configured.")
print(f"Effective batch size   : {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Total training steps   : ~{len(tokenised_train) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps) * training_args.num_train_epochs}")

In [ ]:
# Train the model
# Even with our small dataset, this demonstrates the full training loop.
# On a real dataset with thousands of examples, you'd run this for longer.

print("Starting training...")
print("(Only LoRA adapter weights are being updated - the base model is frozen)")
print()

train_result = trainer.train()

print()
print("Training complete.")
print(f"Final training loss : {train_result.training_loss:.4f}")

### Reading the loss

The loss is just a score for how wrong the model's guesses are. Lower is better. Two numbers to watch while training:

- **Training loss** should fall over time. That means the model is learning.
- **Validation loss**, measured on examples it did not train on, should fall too.

The one to watch out for is when training loss keeps dropping but validation loss starts climbing. That is called overfitting: the model is memorising the exact training examples instead of learning the general pattern, so it does worse on anything new.

With a tiny demonstration dataset like ours, a bit of overfitting is normal and expected. In a real project you would have thousands of examples, which makes it much less of an issue.


## 5. Saving the adapter

Now for one of the nicest payoffs of LoRA. Because we only trained the small add-on grids, that is all we need to save. The next cell saves just the adapter, which is a tiny file, and not the whole multi-gigabyte model.


In [ ]:
# Save the LoRA adapter weights
# Note: we only save the adapter (small), not the full model
# To deploy: load base model + adapter together

adapter_path = MODEL_DIR / "tinyllama-financial-lora" / "final_adapter"
model.save_pretrained(str(adapter_path))
tokeniser.save_pretrained(str(adapter_path))

# Check file sizes
import os
total_size = sum(
    f.stat().st_size
    for f in adapter_path.rglob("*")
    if f.is_file()
)
print(f"Adapter saved to: {adapter_path}")
print(f"Adapter size: {total_size / 1e6:.1f} MB")
print()
print("This is just the LoRA adapter - the small matrices we trained.")
print("The base model (~1.1GB) is downloaded separately and not duplicated.")
print("This means you can share your fine-tuned adapter as a tiny file!")

## 6. Using the fine-tuned model

To actually run the fine-tuned model, you load two things and stack them: the original base model, plus your little adapter on top. The next cell does that and then asks it a financial question so you can see it in action.


In [ ]:
# Load base model + adapter together
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
)
ft_model = PeftModel.from_pretrained(base_model, str(adapter_path))
ft_model = ft_model.to(device)

def generate_ft(prompt, max_new_tokens=150):
    messages = [
        {"role": "system", "content": "You are a financial analyst assistant."},
        {"role": "user", "content": prompt},
    ]
    formatted = tokeniser.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokeniser(formatted, return_tensors="pt").to(ft_model.device)
    
    with torch.no_grad():
        outputs = ft_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokeniser.eos_token_id,
        )
    return tokeniser.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


# Test the fine-tuned model
question = "What were Apple's total net sales in fiscal year 2023?"
print(f"Question: {question}")
print()
print("Fine-tuned model response:")
print("-" * 60)
print(generate_ft(question))

## Recap

In this notebook we:

- Saw why LoRA is what makes fine-tuning possible on a normal computer
- Set up LoRA add-ons on the model's attention layers
- Trained only about 0.5% of the model's numbers and left the other 99.5% frozen
- Ran a full training loop from start to finish
- Saved just the small adapter file rather than a copy of the whole model

The model now behaves differently on our subject. But two questions are still open. Does it actually behave better? And are its answers the kind we actually want? The next two notebooks deal with exactly those.

Next up: notebook 05, Alignment with DPO.
